In [2]:
# ============================================================
# 13_external_validation_leakagefree.ipynb
# Block 1: Load GSE68465 + probe-gene mapping + gene coverage check
# ============================================================

import gzip
import pandas as pd
import numpy as np

base = '/Users/parthshringarpure/Desktop/AI/Projects/luad_survival'

# ---- Load probe -> gene mapping ----
probe_map = pd.read_csv(f'{base}/data/external/GPL96_probe_to_gene.csv')
# Split multi-gene entries, keep first gene as primary mapping
probe_map['gene_symbol_primary'] = probe_map['gene_symbol'].str.split(' /// ').str[0]
probe_to_gene = dict(zip(probe_map['probe_id'], probe_map['gene_symbol_primary']))

print(f"Probe->gene mappings loaded: {len(probe_to_gene)}")

# ---- Parse series matrix: expression + clinical in one pass ----
path = f'{base}/data/external/GSE68465_series_matrix.txt.gz'

sample_ids = None
characteristics_rows = []  # list of (field_name_guess, [values])
expr_rows = []
expr_header = None
in_table = False

with gzip.open(path, 'rt', errors='replace') as f:
    for line in f:
        line = line.rstrip('\n')
        if line.startswith('!Sample_geo_accession'):
            sample_ids = [x.strip('"') for x in line.split('\t')[1:]]
        elif line.startswith('!Sample_characteristics_ch1'):
            vals = [x.strip('"') for x in line.split('\t')[1:]]
            characteristics_rows.append(vals)
        elif line.startswith('!series_matrix_table_begin'):
            in_table = True
            continue
        elif line.startswith('!series_matrix_table_end'):
            in_table = False
            continue
        elif in_table:
            fields = [x.strip('"') for x in line.split('\t')]
            if expr_header is None:
                expr_header = fields  # ["ID_REF", sample1, sample2, ...]
            else:
                expr_rows.append(fields)

print(f"Samples found: {len(sample_ids)}")
print(f"Characteristics rows found: {len(characteristics_rows)}")
print(f"Expression rows (probes) parsed: {len(expr_rows)}")

# ---- Build clinical dataframe from characteristics rows ----
# Each row is one clinical field, values in same order as sample_ids
clinical_ext = pd.DataFrame(index=sample_ids)
for row in characteristics_rows:
    if len(row) != len(sample_ids):
        continue
    # infer field name from "fieldname: value" pattern in first entry
    field_name = row[0].split(':')[0].strip()
    values = [v.split(':', 1)[1].strip() if ':' in v else v for v in row]
    clinical_ext[field_name] = values

print(f"\nClinical fields extracted: {clinical_ext.columns.tolist()}")

# ---- Build expression dataframe (probes x samples) ----
expr_ext = pd.DataFrame(expr_rows, columns=expr_header)
expr_ext = expr_ext.set_index('ID_REF')
expr_ext = expr_ext.apply(pd.to_numeric, errors='coerce')
expr_ext.columns = sample_ids  # align to GSM IDs

print(f"\nExpression matrix shape: {expr_ext.shape}")

# ---- Map probes to genes, collapse to gene-level (mean across probes) ----
expr_ext_gene = expr_ext.copy()
expr_ext_gene['gene'] = expr_ext_gene.index.map(probe_to_gene)
expr_ext_gene = expr_ext_gene.dropna(subset=['gene'])
expr_gene_level = expr_ext_gene.groupby('gene').mean()

print(f"Gene-level expression matrix shape: {expr_gene_level.shape}")

# ---- Gene coverage check against consensus signature ----
# Paste your Block 9 consensus gene list here (>=50% of 50 fits)
consensus_genes = []  # <-- FILL IN from primary notebook's stability_summary
top10_core_genes = ['HOXB9', 'DKK1', 'NLRP2', 'NTS', 'CLEC18A',
                     'LTK', 'C1QL2', 'EREG', 'IGFBP1', 'C20orf114']

available_genes = set(expr_gene_level.index)

print(f"\n--- Top-10 core gene coverage on GSE68465 (GPL96) ---")
for gene in top10_core_genes:
    status = "FOUND" if gene in available_genes else "MISSING"
    print(f"  {gene}: {status}")

n_found = sum(1 for g in top10_core_genes if g in available_genes)
print(f"\n{n_found}/{len(top10_core_genes)} core genes available on this platform")

Probe->gene mappings loaded: 21225
Samples found: 462
Characteristics rows found: 16
Expression rows (probes) parsed: 22283

Clinical fields extracted: ['disease_state', 'Sex', 'age', 'race', 'vital_status', 'clinical_treatment_adjuvant_chemo', 'clinical_treatment_adjuvant_rt', 'disease_stage', 'first_progression_or_relapse', 'months_to_first_progression', 'mths_to_last_clinical_assessment', 'months_to_last_contact_or_death', 'smoking_history', 'surgical_margins', 'organism_part', 'histologic_grade']

Expression matrix shape: (22283, 462)
Gene-level expression matrix shape: (13237, 462)

--- Top-10 core gene coverage on GSE68465 (GPL96) ---
  HOXB9: FOUND
  DKK1: FOUND
  NLRP2: FOUND
  NTS: FOUND
  CLEC18A: MISSING
  LTK: FOUND
  C1QL2: MISSING
  EREG: FOUND
  IGFBP1: FOUND
  C20orf114: MISSING

7/10 core genes available on this platform


In [3]:
consensus_genes = ['LTK', 'TMED7-TICAM2', 'SIX1', 'NCAM2', 'DKK1', 'MS4A1', 'BPIL1',
                    'CLEC18A', 'SPRR1B', 'HOXB9', 'EREG', 'C1QL2', 'NLRP2', 'NTS',
                    'TMEM139', 'IGFBP1', 'PKHD1L1', 'EPGN', 'KLK8', 'TMEM215',
                    'CRHR2', 'CNTN3', 'C20orf114', 'COMP', 'LY6K']

available_genes = set(expr_gene_level.index)

print(f"{'Gene':<16}{'Status'}")
print("-" * 30)
found, missing = [], []
for gene in consensus_genes:
    if gene in available_genes:
        found.append(gene)
        print(f"{gene:<16}FOUND")
    else:
        missing.append(gene)
        print(f"{gene:<16}MISSING")

print(f"\n{len(found)}/{len(consensus_genes)} consensus genes available on GPL96")
print(f"Missing: {missing}")

Gene            Status
------------------------------
LTK             FOUND
TMED7-TICAM2    MISSING
SIX1            FOUND
NCAM2           FOUND
DKK1            FOUND
MS4A1           FOUND
BPIL1           MISSING
CLEC18A         MISSING
SPRR1B          FOUND
HOXB9           FOUND
EREG            FOUND
C1QL2           MISSING
NLRP2           FOUND
NTS             FOUND
TMEM139         MISSING
IGFBP1          FOUND
PKHD1L1         MISSING
EPGN            MISSING
KLK8            FOUND
TMEM215         MISSING
CRHR2           FOUND
CNTN3           MISSING
C20orf114       MISSING
COMP            FOUND
LY6K            MISSING

14/25 consensus genes available on GPL96
Missing: ['TMED7-TICAM2', 'BPIL1', 'CLEC18A', 'C1QL2', 'TMEM139', 'PKHD1L1', 'EPGN', 'TMEM215', 'CNTN3', 'C20orf114', 'LY6K']


In [4]:
print(clinical_ext['disease_stage'].value_counts(dropna=False))

disease_stage
pN0pT2    162
pN0pT1    114
pN1pT2     55
pN2pT2     34
pN1pT1     24
           19
pN0pT3     16
pN2pT1     11
pN0pT4      7
pN1pT3      7
pN2pT3      5
pN2pT4      3
pN1pT4      2
pp          2
pNXpT1      1
Name: count, dtype: int64


In [5]:
print("\n--- vital_status ---")
print(clinical_ext['vital_status'].value_counts(dropna=False))

print("\n--- months_to_last_contact_or_death sample ---")
print(clinical_ext['months_to_last_contact_or_death'].head(10))
print(f"Dtype: {clinical_ext['months_to_last_contact_or_death'].dtype}")
print(f"Unique non-numeric-looking values: {[v for v in clinical_ext['months_to_last_contact_or_death'].unique() if not v.replace('.','').replace('-','').isdigit()][:10]}")


--- vital_status ---
vital_status
Dead     236
Alive    207
          19
Name: count, dtype: int64

--- months_to_last_contact_or_death sample ---
GSM1672281    105.6
GSM1672282     25.2
GSM1672283     64.8
GSM1672284     69.2
GSM1672285     33.9
GSM1672286       83
GSM1672287     69.2
GSM1672288     27.1
GSM1672289      5.8
GSM1672290     50.2
Name: months_to_last_contact_or_death, dtype: object
Dtype: object
Unique non-numeric-looking values: ['na', '']


In [6]:
# ============================================================
# Block 2: Clinical field alignment for GSE68465
# - AJCC 6th-edition pT/pN -> Stage I-III (all M0, resected cohort)
# - months -> days (to match TCGA's day-based survival_time)
# - vital_status -> event (0/1)
# ============================================================

def map_tnm_to_stage(tnm):
    """AJCC 6th edition, resected NSCLC (M0 assumed for all)."""
    if not isinstance(tnm, str) or tnm.strip() in ('', 'pp'):
        return None
    tnm = tnm.strip()

    # Extract N and T values
    import re
    n_match = re.search(r'pN(\d|X)', tnm)
    t_match = re.search(r'pT(\d)', tnm)
    if not n_match or not t_match:
        return None
    n = n_match.group(1)
    t = int(t_match.group(1))

    if n == 'X':
        return None  # nodal status unknown, unmappable
    n = int(n)

    if n == 3:
        return 'Stage III'  # N3 -> IIIB regardless of T
    if t == 4:
        return 'Stage III'  # T4 any N(0-2) -> IIIB
    if n == 2:
        return 'Stage III'  # T1-3N2 -> IIIA
    if n == 1:
        if t == 3:
            return 'Stage III'  # T3N1 -> IIIA
        return 'Stage II'       # T1-2N1 -> IIA/IIB
    if n == 0:
        if t == 3:
            return 'Stage II'   # T3N0 -> IIB
        return 'Stage I'        # T1-2N0 -> IA/IB
    return None

clinical_ext['stage_group'] = clinical_ext['disease_stage'].apply(map_tnm_to_stage)

print("Stage mapping result:")
print(clinical_ext['stage_group'].value_counts(dropna=False))

# ---- Survival time: months -> days ----
def parse_months(x):
    if not isinstance(x, str) or x.strip() in ('na', ''):
        return np.nan
    try:
        return float(x)
    except ValueError:
        return np.nan

clinical_ext['survival_months'] = clinical_ext['months_to_last_contact_or_death'].apply(parse_months)
clinical_ext['survival_time'] = clinical_ext['survival_months'] * 30.4375  # avg days/month

# ---- Event coding ----
clinical_ext['event'] = clinical_ext['vital_status'].map({'Dead': 1, 'Alive': 0})

# ---- Age, gender (already clean) ----
clinical_ext['age'] = pd.to_numeric(clinical_ext['age'], errors='coerce')
clinical_ext['gender'] = (clinical_ext['Sex'].str.lower() == 'male').astype(float)

# ---- Drop patients missing any required field ----
required = ['stage_group', 'survival_time', 'event', 'age', 'gender']
n_before = len(clinical_ext)
clinical_ext_clean = clinical_ext.dropna(subset=required)
n_after = len(clinical_ext_clean)

print(f"\nDropped {n_before - n_after} patients missing required fields.")
print(f"Final GSE68465 cohort: {n_after} patients")
print(f"\nFinal stage distribution:")
print(clinical_ext_clean['stage_group'].value_counts())
print(f"\nEvents: {clinical_ext_clean['event'].sum()} ({clinical_ext_clean['event'].mean()*100:.1f}%)")
print(f"\nSurvival time (days) range: {clinical_ext_clean['survival_time'].min():.0f} - {clinical_ext_clean['survival_time'].max():.0f}")

Stage mapping result:
stage_group
Stage I      276
Stage II      95
Stage III     69
None          22
Name: count, dtype: int64

Dropped 23 patients missing required fields.
Final GSE68465 cohort: 439 patients

Final stage distribution:
stage_group
Stage I      276
Stage II      95
Stage III     68
Name: count, dtype: int64

Events: 235.0 (53.5%)

Survival time (days) range: 1 - 6209
